In [11]:
# Install optuna if not already installed
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'optuna', '-q'])

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score
from torch.optim import Adam
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
import os
import traceback
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')
print('All imports OK')

All imports OK


In [12]:

def correlation_matrix(returns, t, K, eps=0.0, active=None):
    t = pd.to_datetime(t)
    windowed_returns = returns.loc[t - pd.Timedelta(days=K*1.5): t].dropna(how='all')
    window = windowed_returns.dropna(axis=1, how='all')
    if active is not None:
        window = window[active]
    if eps == 0.0:
        window = window.loc[:, ~(window.fillna(0.0) == 0.0).all(axis=0)]
    else:
        window = window.loc[:, ~(window.fillna(0.0).abs() <= eps).all(axis=0)]
    active_cols = window.columns.tolist()
    corr_matrix = window.corr().values
    corr_matrix = np.nan_to_num(corr_matrix, nan=0.0, posinf=0.0, neginf=0.0)
    return corr_matrix, active_cols


def compute_initial_node_embeddings(returns, t, K, embedding_dim=10, eps=0.0, active=None):
    """SVD-based node embeddings. embedding_dim controls how many components to keep."""
    node_embeddings = {}
    t = pd.to_datetime(t)
    windowed_returns = returns.loc[t - pd.Timedelta(days=K*1.5): t].dropna(how='all')
    if windowed_returns.empty:
        return {}, []
    window = windowed_returns.dropna(axis=1, how='all')
    if active is not None:
        valid_active = [c for c in active if c in window.columns]
        if not valid_active:
            return {}, []
        window = window[valid_active]
    if window.shape[0] < 2 or window.shape[1] == 0:
        return {}, []
    if eps == 0.0:
        window = window.loc[:, ~(window.fillna(0.0) == 0.0).all(axis=0)]
    else:
        window = window.loc[:, ~(window.fillna(0.0).abs() <= eps).all(axis=0)]
    active_cols = window.columns.tolist()
    if not active_cols:
        return {}, []
    U, S, Vt = np.linalg.svd(window.values, full_matrices=False)
    V = Vt.T
    H = V @ np.diag(S)
    H = H[:, :embedding_dim]                                   # <-- uses embedding_dim
    if H.shape[1] < embedding_dim:
        padding = np.zeros((H.shape[0], embedding_dim - H.shape[1]))
        H = np.hstack([H, padding])
    scaler = MinMaxScaler(feature_range=(0, 1))
    H = scaler.fit_transform(H)
    for i, stock in enumerate(active_cols):
        node_embeddings[stock] = np.array(H[i, :])
    return node_embeddings, active_cols


def get_active_stocks(returns, t, lookback_days, feature_dfs=None, min_obs=21, eps=0.0):
    t = pd.to_datetime(t)
    window = returns.loc[t - pd.Timedelta(days=lookback_days): t]
    counts = window.notna().sum(axis=0)
    ok_obs = counts >= min_obs
    if eps == 0.0:
        ok_nonzero = ~(window.fillna(0.0) == 0.0).all(axis=0)
    else:
        ok_nonzero = ~(window.fillna(0.0).abs() <= eps).all(axis=0)
    active = window.columns[ok_obs & ok_nonzero].tolist()
    if feature_dfs is not None:
        active_set = set(active)
        for df in feature_dfs:
            if not df.empty:
                active_set = active_set.intersection(df.columns)
        active = list(active_set)
    return active


def create_adjacency_from_correlation(corr_matrix, threshold=0.0):
    C = np.array(corr_matrix, dtype=np.float32)
    A_pos = np.maximum(C, 0.0)
    A_neg = np.maximum(-C, 0.0)
    A_pos[A_pos < threshold] = 0
    A_neg[A_neg < threshold] = 0
    np.fill_diagonal(A_pos, 0)
    np.fill_diagonal(A_neg, 0)
    return torch.tensor(A_pos, dtype=torch.float32), torch.tensor(A_neg, dtype=torch.float32)


def create_edge_index(corr_matrix, k_neighbors=15):
    N = corr_matrix.shape[0]
    if isinstance(corr_matrix, np.ndarray):
        corr = torch.tensor(corr_matrix, dtype=torch.float32)
    else:
        corr = corr_matrix.clone()
    mask_diag = torch.eye(N, dtype=torch.bool, device=corr.device)
    corr.masked_fill_(mask_diag, float('-inf'))
    k = min(k_neighbors, N - 1)
    _, indices = torch.topk(corr.abs(), k=k, dim=1)
    src_list = torch.arange(N, device=corr.device).repeat_interleave(k)
    trg_list = indices.flatten()
    return torch.stack([src_list, trg_list], dim=0)


def prepare_node_features(stocks, sectors, Z_DATA, t, norm_window=63): # 1 quarter
    """
    Normalise each feature per stock against its own rolling history (z-score),
    preserving signal relative to that stock's recent behaviour.
    """
    rows = []
    t = pd.to_datetime(t)
    
    for stock in stocks:
        sector_id = sectors.loc[stock, 'sector_id'] if stock in sectors.index else 0
        
        # Simple lookup instead of rolling calculation
        feats = [sector_id]
        for key in Z_DATA:
            df = Z_DATA[key]
            val = df.loc[t, stock] if stock in df.columns and t in df.index else 0.0
            feats.append(float(val))
            
        rows.append(feats)

    features = np.array(rows, dtype=np.float32)
    return torch.tensor(np.nan_to_num(features), dtype=torch.float32)


print('Helper functions defined')

Helper functions defined


In [13]:

class GATLayer(torch.nn.Module):
    src_nodes_dim = 0
    trg_nodes_dim = 1
    nodes_dim = 0
    head_dim = 2

    def __init__(self, num_in_features, num_out_features, num_of_heads, concat=True,
                 activation=nn.ELU(), dropout_prob=0.1, add_skip_connection=True,
                 bias=True, log_attention_weights=False):
        super().__init__()
        self.num_of_heads = num_of_heads
        self.num_out_features = num_out_features
        self.concat = concat
        self.add_skip_connection = add_skip_connection
        self.linear_proj = nn.Linear(num_in_features, num_of_heads * num_out_features, bias=False)
        self.scoring_fn_target = nn.Parameter(torch.Tensor(1, num_of_heads, num_out_features))
        self.scoring_fn_source = nn.Parameter(torch.Tensor(1, num_of_heads, num_out_features))
        if bias and concat:
            self.bias = nn.Parameter(torch.Tensor(num_of_heads * num_out_features))
        elif bias and not concat:
            self.bias = nn.Parameter(torch.Tensor(num_out_features))
        else:
            self.register_parameter('bias', None)
        if add_skip_connection:
            self.skip_proj = nn.Linear(num_in_features, num_of_heads * num_out_features, bias=False)
        else:
            self.register_parameter('skip_proj', None)
        self.leakyReLU = nn.LeakyReLU(0.2)
        self.activation = activation
        self.dropout = nn.Dropout(p=dropout_prob)
        self.log_attention_weights = log_attention_weights
        self.attention_weights = None
        self.init_params()

    def forward(self, data):
        in_nodes_features, edge_index, corr = data
        num_of_nodes = in_nodes_features.shape[self.nodes_dim]
        assert edge_index.shape[0] == 2
        in_nodes_features = self.dropout(in_nodes_features)
        nodes_features_proj = self.linear_proj(in_nodes_features).view(-1, self.num_of_heads, self.num_out_features)
        nodes_features_proj = self.dropout(nodes_features_proj)
        scores_source = (nodes_features_proj * self.scoring_fn_source).sum(dim=-1)
        scores_target = (nodes_features_proj * self.scoring_fn_target).sum(dim=-1)
        scores_source_lifted, scores_target_lifted, nodes_features_proj_lifted = self.lift(
            scores_source, scores_target, nodes_features_proj, edge_index)
        scores_per_edge = self.leakyReLU(scores_source_lifted + scores_target_lifted)
        src = edge_index[self.src_nodes_dim]
        trg = edge_index[self.trg_nodes_dim]
        corr_e = corr[src, trg]
        mask_pos = (corr_e >= 0).float().unsqueeze(-1)
        mask_neg = (corr_e < 0).float().unsqueeze(-1)
        att_pos = self.neighborhood_aware_softmax(scores_per_edge, trg, num_of_nodes, mask_pos)
        att_neg = self.neighborhood_aware_softmax(scores_per_edge, trg, num_of_nodes, mask_neg)
        attentions_per_edge = torch.cat([att_pos, att_neg], dim=1)
        attentions_per_edge = self.dropout(attentions_per_edge)
        return attentions_per_edge

    def neighborhood_aware_softmax(self, scores_per_edge, trg_index, num_of_nodes, mask):
        scores_per_edge = scores_per_edge - scores_per_edge.max()
        exp_scores = scores_per_edge.exp() * mask
        denom = self.sum_edge_scores_neighborhood_aware(exp_scores, trg_index, num_of_nodes)
        return (exp_scores / (denom + 1e-16)).unsqueeze(-1)

    def sum_edge_scores_neighborhood_aware(self, exp_scores, trg_index, num_of_nodes):
        trg_index_bc = self.explicit_broadcast(trg_index, exp_scores)
        size = list(exp_scores.shape)
        size[self.nodes_dim] = num_of_nodes
        sums = torch.zeros(size, dtype=exp_scores.dtype, device=exp_scores.device)
        sums.scatter_add_(self.nodes_dim, trg_index_bc, exp_scores)
        return sums.index_select(self.nodes_dim, trg_index)

    def lift(self, scores_source, scores_target, nodes_features_matrix_proj, edge_index):
        src_idx = edge_index[self.src_nodes_dim]
        trg_idx = edge_index[self.trg_nodes_dim]
        scores_source = scores_source.index_select(self.nodes_dim, src_idx)
        scores_target = scores_target.index_select(self.nodes_dim, trg_idx)
        nodes_features_proj_lifted = nodes_features_matrix_proj.index_select(self.nodes_dim, src_idx)
        return scores_source, scores_target, nodes_features_proj_lifted

    def explicit_broadcast(self, this, other):
        for _ in range(this.dim(), other.dim()):
            this = this.unsqueeze(-1)
        return this.expand_as(other)

    def init_params(self):
        nn.init.xavier_uniform_(self.linear_proj.weight)
        nn.init.xavier_uniform_(self.scoring_fn_target)
        nn.init.xavier_uniform_(self.scoring_fn_source)
        if self.bias is not None:
            torch.nn.init.zeros_(self.bias)


class GAT(torch.nn.Module):
    def __init__(self, num_of_layers, num_heads_per_layer, num_features_per_layer,
                 add_skip_connection=True, bias=True, dropout=0.1, log_attention_weights=False):
        super().__init__()
        assert num_of_layers == len(num_heads_per_layer) == len(num_features_per_layer) - 1
        num_heads_per_layer = [1] + num_heads_per_layer
        gat_layers = []
        for i in range(num_of_layers):
            layer = GATLayer(
                num_in_features=num_features_per_layer[i] * num_heads_per_layer[i],
                num_out_features=num_features_per_layer[i+1],
                num_of_heads=num_heads_per_layer[i+1],
                concat=True if i < num_of_layers - 1 else False,
                activation=nn.ELU() if i < num_of_layers - 1 else None,
                dropout_prob=dropout,
                add_skip_connection=add_skip_connection,
                bias=bias,
                log_attention_weights=log_attention_weights
            )
            gat_layers.append(layer)
        self.gat_net = nn.Sequential(*gat_layers)

    def forward(self, data):
        return self.gat_net(data)


class DiffusionConvLayer(nn.Module):
    def __init__(self, in_features, out_channels, num_diffusion_steps=1, bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_channels = out_channels
        self.num_diffusion_steps = num_diffusion_steps
        self.theta_pos = nn.Parameter(torch.Tensor(num_diffusion_steps, in_features, out_channels))
        self.theta_neg = nn.Parameter(torch.Tensor(num_diffusion_steps, in_features, out_channels))
        if bias:
            self.bias_pos = nn.Parameter(torch.Tensor(out_channels))
            self.bias_neg = nn.Parameter(torch.Tensor(out_channels))
        else:
            self.register_parameter('bias_pos', None)
            self.register_parameter('bias_neg', None)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.theta_pos)
        nn.init.xavier_uniform_(self.theta_neg)
        if self.bias_pos is not None:
            nn.init.zeros_(self.bias_pos)
        if self.bias_neg is not None:
            nn.init.zeros_(self.bias_neg)

    def forward(self, X_pos, X_neg, A_pos, A_neg):
        N = X_pos.shape[0]
        D_pos = A_pos.sum(dim=1, keepdim=True) + 1e-8
        D_neg = A_neg.sum(dim=1, keepdim=True) + 1e-8
        A_pos_norm = A_pos / D_pos
        A_neg_norm = A_neg / D_neg
        Z_pos = torch.zeros(N, self.out_channels, device=X_pos.device)
        Z_neg = torch.zeros(N, self.out_channels, device=X_neg.device)
        A_power_pos = torch.eye(N, device=X_pos.device)
        A_power_neg = torch.eye(N, device=X_neg.device)
        for s in range(self.num_diffusion_steps):
            Z_pos += A_power_pos @ X_pos @ self.theta_pos[s]
            Z_neg += A_power_neg @ X_neg @ self.theta_neg[s]
            A_power_pos = A_power_pos @ A_pos_norm
            A_power_neg = A_power_neg @ A_neg_norm
        if self.bias_pos is not None:
            Z_pos += self.bias_pos
            Z_neg += self.bias_neg
        return Z_pos, Z_neg


class SpatialEncoder(nn.Module):
    def __init__(self, in_features_dim, hidden_channels, num_diffusion_steps=1, dropout=0.1):
        super().__init__()
        channels = [in_features_dim] + hidden_channels
        layers = []
        for i in range(len(channels) - 1):
            layers.append(DiffusionConvLayer(channels[i], channels[i+1], num_diffusion_steps))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
        self.layers = nn.ModuleList(layers)

    def forward(self, Z_pos, Z_neg, A_pos, A_neg):
        for layer in self.layers:
            if isinstance(layer, DiffusionConvLayer):
                Z_pos, Z_neg = layer(Z_pos, Z_neg, A_pos, A_neg)
            else:
                Z_pos = layer(Z_pos)
                Z_neg = layer(Z_neg)
        return Z_pos, Z_neg


class GraphConvGRUCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_diffusion_steps=1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.conv_r = DiffusionConvLayer(input_dim + hidden_dim, hidden_dim, num_diffusion_steps)
        self.conv_u = DiffusionConvLayer(input_dim + hidden_dim, hidden_dim, num_diffusion_steps)
        self.conv_c = DiffusionConvLayer(input_dim + hidden_dim, hidden_dim, num_diffusion_steps)

    def forward(self, x_t_pos, x_t_neg, h_prev_pos, h_prev_neg, A_pos, A_neg):
        combined_pos = torch.cat([x_t_pos, h_prev_pos], dim=1)
        combined_neg = torch.cat([x_t_neg, h_prev_neg], dim=1)
        r_pos, r_neg = self.conv_r(combined_pos, combined_neg, A_pos, A_neg)
        r_t_pos, r_t_neg = torch.sigmoid(r_pos), torch.sigmoid(r_neg)
        u_pos, u_neg = self.conv_u(combined_pos, combined_neg, A_pos, A_neg)
        u_t_pos, u_t_neg = torch.sigmoid(u_pos), torch.sigmoid(u_neg)
        c_in_pos = torch.cat([x_t_pos, r_t_pos * h_prev_pos], dim=1)
        c_in_neg = torch.cat([x_t_neg, r_t_neg * h_prev_neg], dim=1)
        c_pos, c_neg = self.conv_c(c_in_pos, c_in_neg, A_pos, A_neg)
        c_t_pos, c_t_neg = torch.tanh(c_pos), torch.tanh(c_neg)
        h_t_pos = u_t_pos * h_prev_pos + (1 - u_t_pos) * c_t_pos
        h_t_neg = u_t_neg * h_prev_neg + (1 - u_t_neg) * c_t_neg
        return h_t_pos, h_t_neg

class StructuralAnomalyBranch(nn.Module):
    """
    Dedicated structural encoder — completely decoupled from h.
    Inner product decoder now works because z_struct IS trained to
    encode structural similarity.
    """
    def __init__(self, latent_dim=16):
        super().__init__()
        # 4 fixed-dim structural features per node → latent
        self.encoder = nn.Sequential(
            nn.Linear(4, latent_dim),
            nn.ReLU(),
            nn.Linear(latent_dim, latent_dim)
        )

    def _structural_features(self, A_pos, A_neg):
        N = A_pos.shape[0]
        pos_deg = A_pos.sum(dim=1, keepdim=True) / (N - 1 + 1e-8)   # (N,1)
        neg_deg = A_neg.sum(dim=1, keepdim=True) / (N - 1 + 1e-8)   # (N,1)
        # Local clustering: diag(A^3) / (d*(d-1))
        pos_tri  = torch.diag(A_pos @ A_pos @ A_pos).unsqueeze(1)
        neg_tri  = torch.diag(A_neg @ A_neg @ A_neg).unsqueeze(1)
        pos_clus = pos_tri / (pos_deg * (N - 1) * (pos_deg * (N - 1) - 1) + 1e-8)
        neg_clus = neg_tri / (neg_deg * (N - 1) * (neg_deg * (N - 1) - 1) + 1e-8)
        return torch.cat([pos_deg, neg_deg, pos_clus, neg_clus], dim=1)  # (N, 4)

    def forward(self, A_pos, A_neg):
        feats   = self._structural_features(A_pos, A_neg)      # (N, 4)
        z       = self.encoder(feats)                           # (N, latent_dim)
        A_hat_pos = torch.sigmoid( z @ z.T)                    # (N, N)
        A_hat_neg = torch.sigmoid(-z @ z.T)
        err_pos = torch.norm(A_pos - A_hat_pos, dim=1)         # (N,)
        err_neg = torch.norm(A_neg - A_hat_neg, dim=1)
        struct_error = 0.5 * (err_pos + err_neg) #  
        N = A_pos.shape[0]
        loss = torch.mean(struct_error ** 2) # loss is mean of error squared
        signal = (struct_error - struct_error.min()) / (struct_error.max() - struct_error.min() + 1e-8)
        return loss, signal

class ReconstructionDecoder(nn.Module):
    def __init__(self, hidden_dim, feature_dim, alpha=1.0, use_structure_recon=True):
        """alpha: weight on feature reconstruction loss (0=structure only, 1=feature only)"""
        super().__init__()
        self.use_structure_recon = use_structure_recon
        self.alpha = alpha                                      # <-- now a parameter
        self.feature_decoder = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, feature_dim)
        )

    def forward(self, h_pos, h_neg, X_t, A_t_pos, A_t_neg):
        h_t = h_pos + h_neg
        X_hat = self.feature_decoder(h_t)
        feature_error = torch.norm(X_t - X_hat, dim=1)
        if self.use_structure_recon and A_t_pos is not None and A_t_neg is not None:
            A_hat_pos = torch.sigmoid(h_pos @ h_pos.T)
            A_hat_neg = torch.sigmoid(h_neg @ h_neg.T)
            struct_err_pos = torch.norm(A_t_pos - A_hat_pos, dim=1)
            struct_err_neg = torch.norm(A_t_neg - A_hat_neg, dim=1)
            structure_error = 0.5 * (struct_err_pos + struct_err_neg)
            loss_struct = 0.5 * (torch.mean((A_t_pos - A_hat_pos)**2) +
                                 torch.mean((A_t_neg - A_hat_neg)**2))
            combined_error = self.alpha * feature_error + (1 - self.alpha) * structure_error
            loss_feat = torch.mean(feature_error ** 2) 
            total_loss = self.alpha * loss_feat + (1 - self.alpha) * loss_struct
        else:
            A_hat_pos = A_hat_neg = None
            combined_error = feature_error
            total_loss = torch.mean(feature_error ** 2)
        signals = (combined_error - combined_error.min()) / (combined_error.max() - combined_error.min() + 1e-8)
        return X_hat, A_hat_pos, A_hat_neg, total_loss, signals


class BubbleDetectionModel(nn.Module):
    def __init__(self, gat_config, feature_dim, embedding_dim, encoder_channels,
                 hidden_dim, num_diffusion_steps=1, use_structure_recon=True,
                 alpha=1.0, dropout=0.1):
        super().__init__()
        self.feature_dim = feature_dim
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.alpha = alpha
        self.gat = GAT(
            num_of_layers=gat_config['num_layers'],
            num_heads_per_layer=gat_config['heads'],
            num_features_per_layer=gat_config['features'],
            dropout=dropout
        )
        self.spatial_encoder = SpatialEncoder(
            in_features_dim=feature_dim,
            hidden_channels=encoder_channels,
            num_diffusion_steps=num_diffusion_steps,
            dropout=dropout
        )
        spatial_out_dim = encoder_channels[-1]
        self.gru = GraphConvGRUCell(
            input_dim=spatial_out_dim + embedding_dim,
            hidden_dim=hidden_dim,
            num_diffusion_steps=num_diffusion_steps
        )
        self.struct_branch = StructuralAnomalyBranch(latent_dim=16)
        self.decoder = ReconstructionDecoder(
            hidden_dim=hidden_dim,
            feature_dim=feature_dim,
            alpha=alpha,
            use_structure_recon=use_structure_recon
        )

    def forward(self, X_t, H_t, edge_index, corr_matrix, h_prev_pos=None,
                h_prev_neg=None, A_t_pos=None, A_t_neg=None):
        N = X_t.shape[0]
        if h_prev_pos is None:
            h_prev_pos = torch.zeros(N, self.hidden_dim, device=X_t.device)
        if h_prev_neg is None:
            h_prev_neg = torch.zeros(N, self.hidden_dim, device=X_t.device)
        attention_weights = self.gat((H_t, edge_index, corr_matrix))
        A_pos, A_neg = self.attention_to_adjacency(attention_weights, edge_index, N)
        
        Z_t_pos, Z_t_neg = self.spatial_encoder(X_t, X_t, A_pos, A_neg)

        Z_t_full_pos = torch.cat([Z_t_pos, H_t], dim=1)
        Z_t_full_neg = torch.cat([Z_t_neg, H_t], dim=1)

        h_t_pos, h_t_neg = self.gru(Z_t_full_pos, Z_t_full_neg, h_prev_pos, h_prev_neg, A_pos, A_neg)

        X_hat, A_hat_pos, A_hat_neg, feat_loss, feat_signals = self.decoder(
            h_t_pos, h_t_neg, X_t, A_t_pos, A_t_neg)
        
        struct_loss, struct_signal = self.struct_branch(A_pos, A_neg)

        alpha = self.alpha
        signal_combined = (alpha * feat_signals) + ((1 - alpha) * struct_signal)

        loss = (alpha * feat_loss) + ((1 - alpha) * struct_loss)
        return h_t_pos, h_t_neg, signal_combined, loss, A_pos, A_neg

    def attention_to_adjacency(self, attention_weights, edge_index, num_nodes):
        NH = attention_weights.shape[1] // 2
        att_pos = attention_weights[:, :NH, 0].mean(dim=1)
        att_neg = attention_weights[:, NH:, 0].mean(dim=1)
        src, trg = edge_index[0], edge_index[1]
        A_pos = torch.zeros(num_nodes, num_nodes, device=attention_weights.device)
        A_neg = torch.zeros(num_nodes, num_nodes, device=attention_weights.device)
        A_pos[src, trg] = att_pos
        A_neg[src, trg] = att_neg
        A_pos = (A_pos + A_pos.T) / 2
        A_neg = (A_neg + A_neg.T) / 2
        return A_pos, A_neg

print('Model classes defined')

Model classes defined


In [14]:
# â”€â”€ Training and testing functions â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def train_model(model, optimizer, returns, sectors, volatility, Market_caps, PE_ratios,
                Implied_vol, Short_interest, Beta, Operating_margin, Return_on_equity,
                RSI_momentum, Turnover, Z_DATA, train_dates, stock2idx,
                K=21, embedding_dim=10, corr_threshold=0.0, k_neighbors=15,
                n_epochs=3, device='cpu'):
    N_full = len(stock2idx)
    feature_dfs_list = [volatility, Market_caps, PE_ratios, Implied_vol, Short_interest,
                        Beta, Operating_margin, Return_on_equity, RSI_momentum, Turnover]
    training_losses = []


    for epoch in range(n_epochs):
        h_prev_pos_full = torch.zeros(N_full, model.hidden_dim, dtype=torch.float32)
        h_prev_neg_full = torch.zeros(N_full, model.hidden_dim, dtype=torch.float32)
        epoch_losses = []
        for t in tqdm(train_dates, desc=f'Epoch {epoch+1}/{n_epochs}', leave=False):
            try:
                active = get_active_stocks(returns, t, lookback_days=int(K*1.5),
                                           feature_dfs=feature_dfs_list, min_obs=K)
                if not active:
                    continue
                returns_t          = returns[active]
                # volatility_t       = volatility[active]
                # market_caps_t      = Market_caps[active]
                # PE_ratios_t        = PE_ratios[active]
                # Implied_vol_t      = Implied_vol[active]
                # Short_interest_t   = Short_interest[active]
                # Beta_t             = Beta[active]
                # Operating_margin_t = Operating_margin[active]
                # Return_on_equity_t = Return_on_equity[active]
                # RSI_momentum_t     = RSI_momentum[active]
                # Turnover_t         = Turnover[active]

                node_embeddings, _ = compute_initial_node_embeddings(returns_t, t, K, embedding_dim)
                H_list = []
                for stock in active:
                    emb = node_embeddings.get(stock)
                    if emb is not None:
                        H_list.append(emb[0] if (len(emb.shape) > 1 and emb.shape[0] == 1) else emb)
                    else:
                        H_list.append(np.zeros(embedding_dim))  # <-- uses embedding_dim
                H_t = torch.tensor(np.array(H_list), dtype=torch.float32)

                corr_mat, _ = correlation_matrix(returns_t, t, K)
                corr_t    = torch.tensor(corr_mat, dtype=torch.float32)
                edge_index = create_edge_index(corr_mat, k_neighbors=k_neighbors)
                X_t = prepare_node_features(active, sectors, Z_DATA, t)

                A_t_pos, A_t_neg = create_adjacency_from_correlation(corr_mat, threshold=corr_threshold)

                active_idx = torch.tensor([stock2idx[s] for s in active], dtype=torch.long)
                h_prev_pos = h_prev_pos_full.index_select(0, active_idx)
                h_prev_neg = h_prev_neg_full.index_select(0, active_idx)

                model.train()
                optimizer.zero_grad()
                h_t_pos, h_t_neg, signals, loss, A_pos, A_neg = model(
                    X_t, H_t, edge_index, corr_t,
                    h_prev_pos=h_prev_pos, h_prev_neg=h_prev_neg,
                    A_t_pos=A_t_pos, A_t_neg=A_t_neg
                )
                if torch.isnan(loss):
                    continue
                loss.backward()
                optimizer.step()

                h_prev_pos_full[active_idx] = h_t_pos.detach()
                h_prev_neg_full[active_idx] = h_t_neg.detach()
                epoch_losses.append(loss.item())
            except Exception:
                continue

        avg = np.mean(epoch_losses) if epoch_losses else float('nan')
        training_losses.extend(epoch_losses)
        print(f'  Epoch {epoch+1} avg loss: {avg:.4f}')
    return training_losses


def test_model(model, returns, sectors, volatility, Market_caps, PE_ratios, Implied_vol, Short_interest,
                        Beta, Operating_margin, Return_on_equity, RSI_momentum, Turnover,Z_DATA, test_dates, stock2idx,
               K=21, embedding_dim=10, corr_threshold=0.0, k_neighbors=15):
    N_full = len(stock2idx)
    h_prev_pos_full = torch.zeros(N_full, model.hidden_dim)
    h_prev_neg_full = torch.zeros(N_full, model.hidden_dim)
    feature_dfs_list = [volatility, Market_caps, PE_ratios, Implied_vol, Short_interest,
                        Beta, Operating_margin, Return_on_equity, RSI_momentum, Turnover]
    names = ['volatility', 'Market_caps', 'PE_ratios', 'Implied_vol', 'Short_interest',
             'Beta', 'Operating_margin', 'Return_on_equity', 'RSI_momentum', 'Turnover']
    all_signals, valid_dates, active_lists = [], [], []

    model.eval()
    for t in tqdm(test_dates, desc='Testing', leave=False):
        try:
            active = get_active_stocks(returns, t, lookback_days=int(K*1.5),
                                       feature_dfs=feature_dfs_list, min_obs=K)
            if not active:
                continue
            
            returns_t          = returns[active]
            # volatility_t       = volatility[active]
            # market_caps_t      = Market_caps[active]
            # PE_ratios_t        = PE_ratios[active]
            # Implied_vol_t      = Implied_vol[active]
            # Short_interest_t   = Short_interest[active]
            # Beta_t             = Beta[active]
            # Operating_margin_t = Operating_margin[active]
            # Return_on_equity_t = Return_on_equity[active]
            # RSI_momentum_t     = RSI_momentum[active]
            # Turnover_t         = Turnover[active]

            node_embeddings, _ = compute_initial_node_embeddings(returns_t, t, K, embedding_dim)
            H_list = []
            for stock in active:
                emb = node_embeddings.get(stock)
                H_list.append(emb if emb is not None else np.zeros(embedding_dim))
            H_t = torch.tensor(np.array(H_list), dtype=torch.float32)

            corr_mat, _ = correlation_matrix(returns_t, t, K)
            corr_t     = torch.tensor(corr_mat, dtype=torch.float32)
            edge_index = create_edge_index(corr_mat, k_neighbors=k_neighbors)
            X_t = prepare_node_features(active, sectors, Z_DATA, t)


            active_idx = torch.tensor([stock2idx[s] for s in active], dtype=torch.long)
            h_prev_pos = h_prev_pos_full.index_select(0, active_idx)
            h_prev_neg = h_prev_neg_full.index_select(0, active_idx)

            with torch.no_grad():
                h_t_pos, h_t_neg, signals, _, A_pos, A_neg = model(
                    X_t, H_t, edge_index, corr_t,
                    h_prev_pos=h_prev_pos, h_prev_neg=h_prev_neg,
                    A_t_pos=None, A_t_neg=None
                )
            h_prev_pos_full[active_idx] = h_t_pos.detach()
            h_prev_neg_full[active_idx] = h_t_neg.detach()
            all_signals.append(signals.cpu().numpy())
            valid_dates.append(t)
            active_lists.append(active)
        except Exception:
            continue

    return {d: (active_lists[i], all_signals[i]) for i, d in enumerate(valid_dates)}


def evaluate_auc(test_results, prices, forward_window=5, crash_threshold=-0.20):
    y_true, y_scores = [], []
    sorted_dates = sorted(test_results.keys())
    valid_dates = [d for d in sorted_dates
                   if d <= prices.index[-1] - pd.Timedelta(days=forward_window)]
    for t in valid_dates:
        stocks, signals = test_results[t]
        if isinstance(signals, torch.Tensor):
            signals = signals.cpu().numpy()
        signals = signals.flatten()
        available = [s for s in stocks if s in prices.columns]
        if not available:
            continue
        mask = [i for i, s in enumerate(stocks) if s in available]
        signals = signals[mask]
        p_t = prices.loc[t, available]
        future_idx = prices.index.searchsorted(t + pd.Timedelta(days=forward_window))
        if future_idx >= len(prices):
            continue
        p_future = prices.loc[prices.index[future_idx], available]
        fwd_returns = (p_future - p_t) / p_t
        crash = (fwd_returns < crash_threshold).astype(int)
        y_true.extend(crash.values)
        y_scores.extend(signals)
    if len(y_true) == 0 or len(np.unique(y_true)) < 2:
        return None
    return roc_auc_score(np.array(y_true), np.array(y_scores))

print('Train / test / evaluate functions defined')

Train / test / evaluate functions defined


In [15]:
# â”€â”€ Load all data once (expensive - only do this once) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def load_and_fix_index(filename):
    df = pd.read_csv(filename, index_col=0)
    df.index = pd.to_datetime(df.index, format='%m/%d/%Y')
    df.dropna(how='all', inplace=True)
    #df = df.ffill().bfill()
    return df

print('[1/3] Loading prices and returns...')
prices_raw = pd.read_excel('SPX_sectors_data.xlsx', header=[0,1], index_col=0)
prices_raw.dropna(how='all', inplace=True)
prices_raw = prices_raw.ffill().bfill()
prices_raw.columns = prices_raw.columns.droplevel(1)

all_stocks = prices_raw.columns.tolist()
stock2idx  = {s: i for i, s in enumerate(all_stocks)}
N_full     = len(all_stocks)

sectors = pd.read_excel('SPX_sectors_data.xlsx', sheet_name='Sectors', header=0, index_col=0)
sectors['sector_id'] = sectors['Sector'].astype('category').cat.codes

returns_raw = pd.read_excel('SPX_sectors_data.xlsx', header=[0,1], index_col=0)
returns_raw.columns = returns_raw.columns.get_level_values(0)
returns_raw.dropna(how='all', inplace=True)
returns_raw = returns_raw.pct_change().dropna(how='all').ffill().bfill()
volatility = returns_raw.rolling(window=21).std().dropna(how='all') * np.sqrt(252)

train_returns = returns_raw.loc["2012-01-01":"2016-12-31"]
test_returns  = returns_raw.loc['2019-07-01':'2024-12-31']
test_prices   = prices_raw.loc['2019-07-01':'2024-12-31']
val_prices = prices_raw.loc["2017-01-01":"2019-06-30"]

print('[2/3] Loading factor data...')
Market_caps      = load_and_fix_index('Data/SPX_Constituents_market_cap_2006_2025(in).csv')
PE_ratios        = load_and_fix_index('Data/SPX_Constituents_Calculated_PE_2006_2025(in).csv')
Implied_vol      = load_and_fix_index('Data/SPX_Constituents_Implied_vol_2006_2025(in).csv')
Beta             = load_and_fix_index('Data/SPX_Constituents_Beta_2006_2025(in).csv')
Operating_margin = load_and_fix_index('Data/SPX_Constituents_Op_Margin_2006_2025(in).csv')
Return_on_equity = load_and_fix_index('Data/SPX_Constituents_Ret_On_Equity_2006_2025(in).csv')
RSI_momentum     = load_and_fix_index('Data/SPX_Constituents_RSI_momentum_2006_2025(in).csv')
Short_interest   = load_and_fix_index('Data/SPX_Constituents_Short_Interest_Pct_2006_2025(in).csv')
Turnover         = load_and_fix_index('Data/SPX_Constituents_Turnover_30D_2006_2025(in).csv')

print('[3/3] Data loaded. Ready to tune.')

[1/3] Loading prices and returns...
[2/3] Loading factor data...
[3/3] Data loaded. Ready to tune.


In [16]:
print('Precomputing Z scores for all features...')

def precompute_zscores(df, window=63):
    """Calculates rolling z-scores for an entire dataframe at once."""
    rolling_mean = df.rolling(window=window, min_periods=5).mean()
    rolling_std = df.rolling(window=window, min_periods=5).std()
    # Avoid division by zero with 1e-8
    z_scores = (df - rolling_mean) / (rolling_std + 1e-8)
    # Clip outliers to keep gradients stable and fill NaNs
    return z_scores.clip(-5.0, 5.0).fillna(0.0)

# Precompute z-scores for all feature dataframes
Z_DATA = {
    'volatility':       precompute_zscores(volatility), # use your vol_window here
    'market_caps':      precompute_zscores(Market_caps),
    'pe_ratios':        precompute_zscores(PE_ratios),
    'implied_vol':      precompute_zscores(Implied_vol),
    'short_interest':   precompute_zscores(Short_interest),
    'beta':             precompute_zscores(Beta),
    'op_margin':        precompute_zscores(Operating_margin),
    'roe':              precompute_zscores(Return_on_equity),
    'rsi':              precompute_zscores(RSI_momentum),
    'turnover':         precompute_zscores(Turnover)
}

Precomputing Z scores for all features...


In [ ]:
# ── Optuna objective ───────────────────────────────────────────────────────────
# forward_window and crash_threshold are tuned alongside model hyperparameters.
# The objective is the AUC at the sampled (forward_window, crash_threshold) pair,
# so Optuna finds params that work well across different evaluation regimes.
#
# Edit these lists to control which eval settings are searched over.
EVAL_FORWARD_WINDOWS  = [22]           # days ahead to look for a crash
EVAL_CRASH_THRESHOLDS = [-0.20, -0.30] # return threshold to define crash

def objective(trial):

    # ── Data hyperparameters ──────────────────────────────────────────────────
    K             = trial.suggest_int('K', 10, 42)
    embedding_dim = trial.suggest_int('embedding_dim', 5, 20)
    corr_threshold= trial.suggest_float('corr_threshold', 0.0, 0.5)
    k_neighbors   = trial.suggest_int('k_neighbors', 5, 30)
    vol_window    = trial.suggest_int('vol_window', 10, 42)

    # ── Architecture hyperparameters ─────────────────────────────────────────
    gat_heads       = trial.suggest_int('gat_heads', 1, 4)
    gat_out_dim     = trial.suggest_int('gat_out_dim', 4, 32)
    n_enc_layers    = trial.suggest_int('n_encoder_layers', 1, 3)
    enc_dim         = trial.suggest_int('encoder_dim', 8, 64)
    encoder_channels= [enc_dim] * n_enc_layers
    hidden_dim      = trial.suggest_int('hidden_dim', 16, 128)
    n_diff_steps    = trial.suggest_int('num_diffusion_steps', 1, 3)
    dropout         = trial.suggest_float('dropout', 0.0, 0.5)
    alpha           = trial.suggest_float('alpha', 0.0, 1.0)  

    # ── Training hyperparameters ─────────────────────────────────────────────
    lr           = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    n_epochs     = 5

    # ── Evaluation hyperparameters (also tuned) ───────────────────────────────
    forward_window  = trial.suggest_categorical('forward_window',  EVAL_FORWARD_WINDOWS)
    crash_threshold = trial.suggest_categorical('crash_threshold', EVAL_CRASH_THRESHOLDS)

    # ── Recompute volatility with this trial's window ─────────────────────────
    train_volatility = train_returns.rolling(window=vol_window).std().dropna(how='all') * np.sqrt(252)
    test_volatility  = test_returns.rolling(window=vol_window).std().dropna(how='all')  * np.sqrt(252)

    # ── Training / test dates ─────────────────────────────────────────────────
    min_date    = train_returns.index[0] + pd.Timedelta(days=K*2)
    train_dates = train_returns.loc[min_date:].index
    test_min    = test_returns.index[0]  + pd.Timedelta(days=K*2)
    test_dates  = test_returns.loc[test_min:].index

    # ── Build model ───────────────────────────────────────────────────────────
    gat_config = {
        'num_layers': 1,
        'heads'     : [gat_heads],
        'features'  : [embedding_dim, gat_out_dim]
    }
    import torch
    import random
    seed = 0
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    model = BubbleDetectionModel(
        gat_config=gat_config,
        feature_dim=11,
        embedding_dim=embedding_dim,
        encoder_channels=encoder_channels,
        hidden_dim=hidden_dim,
        num_diffusion_steps=n_diff_steps,
        use_structure_recon=True,
        alpha=alpha,
        dropout=dropout
    )
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    # ── Train ─────────────────────────────────────────────────────────────────
    train_model(
        model, optimizer, train_returns, sectors, train_volatility,
        Market_caps, PE_ratios, Implied_vol, Short_interest, Beta,
        Operating_margin, Return_on_equity, RSI_momentum, Turnover, Z_DATA,
        train_dates, stock2idx,
        K=K, embedding_dim=embedding_dim, corr_threshold=corr_threshold,
        k_neighbors=k_neighbors, n_epochs=n_epochs
    )

    # ── Test (run once, reuse for all eval combos) ────────────────────────────
    test_results = test_model(
        model, test_returns, sectors, test_volatility,
        Market_caps, PE_ratios, Implied_vol, Short_interest, Beta,
        Operating_margin, Return_on_equity, RSI_momentum, Turnover, Z_DATA,
        test_dates, stock2idx,
        K=K, embedding_dim=embedding_dim, corr_threshold=corr_threshold,
        k_neighbors=k_neighbors
    )

    # ── Evaluate at sampled (forward_window, crash_threshold) ─────────────────
    auc = evaluate_auc(test_results, test_prices,
                       forward_window=forward_window,
                       crash_threshold=crash_threshold)
    if auc is None:
        return 0.5

    print(f'  Trial {trial.number:3d} | AUC={auc:.4f} | '
          f'fw={forward_window}d  ct={crash_threshold*100:.0f}% | '
          f'K={K}, emb={embedding_dim}, hidden={hidden_dim}, '
          f'enc={encoder_channels}, heads={gat_heads}, '
          f'lr={lr:.5f}, epochs={n_epochs}')
    return auc


In [18]:
# ── Run the study ──────────────────────────────────────────────────────────────
# n_trials: how many configs to try. Each trial trains + tests the full model.
# On CPU with 1 epoch each, expect ~10-20 min per trial.
# Start with n_trials=10 to get a quick feel, then increase.

N_TRIALS = 10

study = optuna.create_study(direction='maximize',
                             study_name='bubble_gae_tuning',
                             sampler=optuna.samplers.TPESampler(seed=0))

# Seed with your current known-good config as trial 0
study.enqueue_trial({
    'K': 28, 'embedding_dim': 16, 'corr_threshold': 0.3, 'k_neighbors': 19,
    'vol_window': 23, 'gat_heads': 3, 'gat_out_dim': 16,
    'n_encoder_layers': 3, 'encoder_dim': 62,
    'hidden_dim': 59, 'num_diffusion_steps': 3, 'dropout': 0.26,
    'alpha': 0.57, 'lr': 0.0071, 'weight_decay': 1.63e-06, 'n_epochs': 5,
    'forward_window': 22, 'crash_threshold': -0.30
})

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
print('\nStudy complete!')


  0%|          | 0/10 [00:00<?, ?it/s]

  Epoch 1 avg loss: 57.4690


  Epoch 2 avg loss: 57.1030


  Epoch 3 avg loss: 56.9446


  Epoch 4 avg loss: 56.4527


  Epoch 5 avg loss: 56.3134


  Epoch 6 avg loss: 56.0704


  Epoch 7 avg loss: 56.1701


[W 2026-04-04 09:52:23,619] Trial 0 failed with parameters: {'K': 28, 'embedding_dim': 16, 'corr_threshold': 0.3, 'k_neighbors': 19, 'vol_window': 23, 'gat_heads': 3, 'gat_out_dim': 16, 'n_encoder_layers': 3, 'encoder_dim': 62, 'hidden_dim': 59, 'num_diffusion_steps': 3, 'dropout': 0.26, 'alpha': 0.57, 'lr': 0.0071, 'weight_decay': 1.63e-06, 'forward_window': 22, 'crash_threshold': -0.3} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "C:\Users\archi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\archi\AppData\Local\Temp\ipykernel_29168\672227584.py", line 75, in objective
    train_model(
    ~~~~~~~~~~~^
        model, optimizer, train_returns, sectors, train_volatility,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<4 lines>...
   

KeyboardInterrupt: 

In [ ]:


print(f'Best AUC: {study.best_value:.4f}')
print(f'Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

# Full results table sorted by AUC
results_df = study.trials_dataframe(attrs=('number', 'value', 'params', 'state'))
results_df = results_df.sort_values('value', ascending=False)
print('\nAll trials:')
display(results_df.head(20))

Best AUC: 0.7329
Best params:
  K: 36
  embedding_dim: 5
  corr_threshold: 0.12595555098526362
  k_neighbors: 15
  vol_window: 20
  gat_heads: 4
  gat_out_dim: 30
  n_encoder_layers: 2
  encoder_dim: 31
  hidden_dim: 19
  num_diffusion_steps: 3
  dropout: 0.031411223157346096
  alpha: 0.6453689865411119
  lr: 0.0008710436515376588
  weight_decay: 0.000826656836336226
  forward_window: 10
  crash_threshold: -0.2

All trials:


,number,value,params_K,params_alpha,params_corr_threshold,params_crash_threshold,params_dropout,params_embedding_dim,params_encoder_dim,params_forward_window,params_gat_heads,params_gat_out_dim,params_hidden_dim,params_k_neighbors,params_lr,params_n_encoder_layers,params_num_diffusion_steps,params_vol_window,params_weight_decay,state
17,17,0.732912,36,0.645369,0.125956,-0.2,0.031411,5,31,10,4,30,19,15,0.000871,2,3,20,0.000827,COMPLETE
14,14,0.637974,27,0.785100,0.136114,-0.3,0.171431,5,64,10,4,23,16,11,0.004124,2,3,21,0.000952,COMPLETE
13,13,0.621775,28,0.819635,0.144410,-0.2,0.162759,7,29,10,4,11,44,11,0.004298,2,2,24,0.000003,COMPLETE
1,1,0.613337,28,0.568045,0.301382,-0.3,0.264447,16,62,10,3,16,59,19,0.007099,3,3,23,0.000002,COMPLETE
3,3,0.598230,21,0.988374,0.348816,-0.2,0.219301,11,25,22,3,10,57,6,0.000160,1,2,32,0.000004,COMPLETE
0,0,0.586171,28,0.570000,0.300000,-0.3,0.260000,16,62,22,3,16,59,19,0.007100,3,3,23,0.000002,COMPLETE
18,18,0.581781,36,0.659164,0.076363,-0.3,0.017493,5,54,10,1,32,16,16,0.000777,2,3,20,0.000733,COMPLETE
16,16,0.575607,30,0.847585,0.013356,-0.3,0.161421,8,17,10,4,23,40,10,0.003879,2,2,26,0.000080,COMPLETE
2,2,0.575006,42,0.456150,0.230740,-0.2,0.387117,17,37,10,3,8,62,25,0.001370,3,1,13,0.000001,COMPLETE
5,5,0.571676,20,0.131798,0.032074,-0.2,0.333705,11,40,22,2,19,121,23,0.002708,1,1,28,0.000007,COMPLETE


In [ ]:
# ── Retrain best config with full epochs ───────────────────────────────────────
# Run this after the study to get a final model using the best hyperparameters.

bp = study.best_params
print(f'Best AUC: {study.best_value:.4f}')
print(f'Best forward_window: {bp["forward_window"]} days')
print(f'Best crash_threshold: {bp["crash_threshold"]*100:.0f}%')
print(f'Best params: {bp}')

encoder_channels_best = [bp['encoder_dim']] * bp['n_encoder_layers']
train_vol_best = train_returns.rolling(window=bp['vol_window']).std().dropna(how='all') * np.sqrt(252)
test_vol_best  = test_returns.rolling(window=bp['vol_window']).std().dropna(how='all')  * np.sqrt(252)

min_date    = train_returns.index[0] + pd.Timedelta(days=bp['K']*2)
train_dates = train_returns.loc[min_date:].index
test_min    = test_returns.index[0]  + pd.Timedelta(days=bp['K']*2)
test_dates  = test_returns.loc[test_min:].index

gat_config_best = {
    'num_layers': 1,
    'heads'     : [bp['gat_heads']],
    'features'  : [bp['embedding_dim'], bp['gat_out_dim']]
}

best_model = BubbleDetectionModel(
    gat_config=gat_config_best,
    feature_dim=11,
    embedding_dim=bp['embedding_dim'],
    encoder_channels=encoder_channels_best,
    hidden_dim=bp['hidden_dim'],
    num_diffusion_steps=bp['num_diffusion_steps'],
    use_structure_recon=True,
    alpha=bp['alpha'],
    dropout=bp['dropout']
)
best_optimizer = Adam(best_model.parameters(), lr=bp['lr'], weight_decay=bp['weight_decay'])

FINAL_EPOCHS = 5  # train longer for the final model

print(f'\nRetraining best config for {FINAL_EPOCHS} epochs...')
train_model(
    best_model, best_optimizer, train_returns, sectors, train_vol_best,
    Market_caps, PE_ratios, Implied_vol, Short_interest, Beta,
    Operating_margin, Return_on_equity, RSI_momentum, Turnover,
    train_dates, stock2idx,
    K=bp['K'], embedding_dim=bp['embedding_dim'],
    corr_threshold=bp['corr_threshold'], k_neighbors=bp['k_neighbors'],
    n_epochs=FINAL_EPOCHS
)

best_test_results = test_model(
    best_model, test_returns, sectors, test_vol_best,
    Market_caps, PE_ratios, Implied_vol, Short_interest, Beta,
    Operating_margin, Return_on_equity, RSI_momentum, Turnover,
    test_dates, stock2idx,
    K=bp['K'], embedding_dim=bp['embedding_dim'],
    corr_threshold=bp['corr_threshold'], k_neighbors=bp['k_neighbors']
)

# Evaluate at the best (forward_window, crash_threshold) found by the study
final_auc = evaluate_auc(best_test_results, test_prices,
                         forward_window=bp['forward_window'],
                         crash_threshold=bp['crash_threshold'])
print(f'\nFinal model AUC (fw={bp["forward_window"]}d, ct={bp["crash_threshold"]*100:.0f}%, {FINAL_EPOCHS} epochs): {final_auc:.4f}')

# Also show AUC across all eval combos so you can see the full picture
print('\nAUC across all (forward_window, crash_threshold) combinations:')
print(f'{"fw":>6} {"ct":>8} {"AUC":>8}')
for fw in EVAL_FORWARD_WINDOWS:
    for ct in EVAL_CRASH_THRESHOLDS:
        auc = evaluate_auc(best_test_results, test_prices, forward_window=fw, crash_threshold=ct)
        marker = ' <-- best' if fw == bp['forward_window'] and ct == bp['crash_threshold'] else ''
        print(f'{fw:>6} {ct*100:>7.0f}% {auc:.4f}{marker}' if auc else f'{fw:>6} {ct*100:>7.0f}%    N/A')

os.makedirs('outputs', exist_ok=True)
pd.to_pickle(best_test_results, 'outputs/test_results_tuned_best.pkl')
pd.to_pickle(test_prices,       'outputs/test_prices_tuned_best.pkl')
torch.save(best_model.state_dict(), 'outputs/best_model_tuned.pt')
print('\nResults saved to outputs/')


Best AUC: 0.7329
Best forward_window: 10 days
Best crash_threshold: -20%
Best params: {'K': 36, 'embedding_dim': 5, 'corr_threshold': 0.12595555098526362, 'k_neighbors': 15, 'vol_window': 20, 'gat_heads': 4, 'gat_out_dim': 30, 'n_encoder_layers': 2, 'encoder_dim': 31, 'hidden_dim': 19, 'num_diffusion_steps': 3, 'dropout': 0.031411223157346096, 'alpha': 0.6453689865411119, 'lr': 0.0008710436515376588, 'weight_decay': 0.000826656836336226, 'forward_window': 10, 'crash_threshold': -0.2}

Retraining best config for 5 epochs...


TypeError: train_model() missing 1 required positional argument: 'stock2idx'